# Agent Identity, Authorization, and Governance

An agent needs a traceable identity and limited authority so every action can be attributed, reviewed, and revoked.


## What to learn

- Authentication establishes who the user, agent, and service are.
- Authorization decides what each identity may do to which resource.
- Delegation grants temporary, scoped authority on behalf of a user.
- Audit logs record the actor, authority, input, action, result, and time.

## Decision rule

Give every agent and workload its own identity. Use short-lived, narrowly scoped credentials; preserve the user-agent delegation chain; require approval for high-impact actions; and provide immediate revocation and kill switches.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
# Import the dependencies used by this example.
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.tools import tool

@dataclass
# Define the data shape and small operations before running them.
class IdentityContext:
    subject: str
    scopes: set[str]

@tool
def read_account(account_id: str) -> str:
    """Read an account."""
    return f"account:{account_id}"

@tool
def transfer_funds(account_id: str, amount: float) -> str:
    """Transfer funds from an account."""
    return f"transferred:{amount}"

@wrap_model_call
def enforce_scopes(request: ModelRequest, handler) -> ModelResponse:
    scopes = request.runtime.context.scopes
    allowed = {"read_account"}
    if "funds:transfer" in scopes:
        allowed.add("transfer_funds")
    return handler(request.override(tools=[tool for tool in request.tools if tool.name in allowed]))

# Configure the framework object; this line prepares it but may not execute it yet.
agent = create_agent(
    model="openai:gpt-5.6-sol",
    tools=[read_account, transfer_funds],
    middleware=[enforce_scopes],
    context_schema=IdentityContext,
)

# Execute the configured model or workflow with the input below.
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Read account A-17"}]},
    context=IdentityContext(subject="user-42", scopes={"accounts:read"}),
)
result["messages"][-1].content


In [ ]:
"""Agent identity broker: scoped, short-lived tokens with delegation chains and
audited attribution (user -> agent -> sub-agent -> tool). No API needed — the broker,
agents, and tools are simulated with pure Python stdlib."""
# Import the dependencies used by this example.
import uuid
from dataclasses import dataclass

CLOCK = [0.0]      # simulated seconds; advanced manually
MAX_TTL = 300      # broker caps every token at 5 minutes
MAX_DEPTH = 3      # max agents allowed in one delegation chain

POLICY = {         # least-privilege grants: agent -> tool -> allowed scopes
    "planner":  {"crm": {"crm:read"}},
    "research": {"web": {"web:search"}, "crm": {"crm:read"}},
    "billing":  {"payments": {"charges:read"}},
}

@dataclass(frozen=True)
# Define the data shape and small operations before running them.
class Token:
    jti: str            # token id
    tool: str           # audience (RFC 8707-style resource binding)
    scopes: frozenset
    chain: tuple        # attribution chain; root is always a human principal
    exp: float

class Broker:
    def __init__(self):
        self.audit = []

    def _log(self, event, chain, **kw):
        self.audit.append({"t": CLOCK[0], "event": event,
                           "chain": " -> ".join(chain), **kw})

    def mint(self, agent, tool, scopes, user=None, parent=None):
        """Issue a scoped short-lived token; passing a parent token = delegation."""
        root = parent.chain if parent else (f"user:{user}",)
        chain = root + (f"agent:{agent}",)
        scopes = frozenset(scopes)
        allowed = POLICY.get(agent, {}).get(tool, frozenset())
        if sum(p.startswith("agent:") for p in chain) > MAX_DEPTH:
            self._log("MINT_DENIED", chain, reason="delegation chain too deep")
            return None
        if not scopes <= allowed:
            self._log("MINT_DENIED", chain,
                      reason=f"scopes {sorted(scopes - allowed)} not granted for '{tool}'")
            return None
        tok = Token(uuid.uuid4().hex[:8], tool, scopes, chain, CLOCK[0] + MAX_TTL)
        self._log("TOKEN_ISSUED", chain, jti=tok.jti, tool=tool,
                  scopes=sorted(scopes), ttl=MAX_TTL)
        return tok

    def invoke(self, tok, tool, scope, action):
        """Tool-side enforcement: expiry, audience, scope — then attributed logging."""
        chain = tok.chain + (f"tool:{tool}",)
        if CLOCK[0] >= tok.exp:
            reason = "token expired (short TTL is a passive kill switch)"
        elif tok.tool != tool:
            reason = f"audience mismatch: token bound to '{tok.tool}' (RFC 8707)"
        elif scope not in tok.scopes:
            reason = f"insufficient_scope: needs '{scope}'"
        else:
            self._log("ACTION_OK", chain, jti=tok.jti, action=action)
            return True
        self._log("ACTION_DENIED", chain, jti=tok.jti, action=action, reason=reason)
        return False

broker = Broker()

# 1. Alice's planner agent gets a CRM read token and uses it.
t_plan = broker.mint("planner", "crm", {"crm:read"}, user="alice")
# Execute the configured model or workflow with the input below.
broker.invoke(t_plan, "crm", "crm:read", "list_accounts")

# 2. Planner delegates to a research sub-agent (chain grows; root stays alice).
t_web = broker.mint("research", "web", {"web:search"}, parent=t_plan)
broker.invoke(t_web, "web", "web:search", "search('acme quarterly filings')")

# 3. Scope violation: research asks for crm:write it was never granted.
broker.mint("research", "crm", {"crm:read", "crm:write"}, parent=t_plan)

# 4. Audience misuse: the web token is replayed against the CRM tool.
broker.invoke(t_web, "crm", "crm:read", "read_contacts")

# 5. TTL expiry: six minutes pass; the old token is dead by design.
CLOCK[0] += 360
broker.invoke(t_plan, "crm", "crm:read", "list_accounts")

# 6. Delegation depth: a 4th-hop sub-sub-sub-agent is refused.
t2 = broker.mint("research", "crm", {"crm:read"}, parent=t_plan)
t3 = broker.mint("planner", "crm", {"crm:read"}, parent=t2)
broker.mint("billing", "payments", {"charges:read"}, parent=t3)

print("=== AUDIT LOG (every row carries a human-rooted attribution chain) ===")
for row in broker.audit:
    extra = {k: v for k, v in row.items() if k not in ("t", "event", "chain")}
    print(f"[t={row['t']:>4.0f}s] {row['event']:<13} {row['chain']}  {extra}")

denied = [r for r in broker.audit if r["event"].endswith("DENIED")]
print("\n=== COMPLIANCE REPORT ===")
print(f"events={len(broker.audit)}  "
      f"tokens_issued={sum(r['event'] == 'TOKEN_ISSUED' for r in broker.audit)}  "
      f"actions_ok={sum(r['event'] == 'ACTION_OK' for r in broker.audit)}  "
      f"denied={len(denied)}")
rooted = sum(r["chain"].startswith("user:") for r in broker.audit)
print(f"attribution coverage: {rooted}/{len(broker.audit)} rows trace to a human principal")
print("denial reasons (your intrusion + policy-tuning signal):")
for r in denied:
    print(f"  - [{r['event']}] {r['reason']}")


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
